In [0]:
# /Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/FactGapsInCare/FactGapsInCare/caregap_analyzer

import os
from datetime import datetime
import pandas as pd


def calculate_care_gaps(
    member_df: pd.DataFrame,
    claims_df: pd.DataFrame,
    lookup_df: pd.DataFrame,
    measurement_year: int = 2026,
) -> pd.DataFrame:
    if member_df.empty or lookup_df.empty:
        return pd.DataFrame()

    # -------------------------------------------------------------
    # 1. Lowercase all column names (e.g. ServiceStartDate -> servicestartdate)
    # -------------------------------------------------------------
    member = member_df.copy()
    member.columns = member.columns.str.strip().str.lower()

    claims = claims_df.copy()
    if not claims.empty:
        claims.columns = claims.columns.str.strip().str.lower()

    lookup = lookup_df.copy()
    lookup.columns = lookup.columns.str.strip().str.lower()

    # -------------------------------------------------------------
    # 2. Convert Lookup Dates & Codes
    # -------------------------------------------------------------
    if "servicestartdate" in lookup.columns:
        lookup["servicestartdate"] = pd.to_datetime(lookup["servicestartdate"])
    else:
        lookup["servicestartdate"] = pd.to_datetime(f"{measurement_year}-01-01")

    if "serviceenddate" in lookup.columns:
        lookup["serviceenddate"] = pd.to_datetime(lookup["serviceenddate"])
    else:
        lookup["serviceenddate"] = pd.to_datetime(f"{measurement_year}-12-31")

    lookup["code"] = lookup["code"].astype(str).str.strip()

    # -------------------------------------------------------------
    # 3. Process Member & Claims
    # -------------------------------------------------------------
    if "dateofbirth" in member.columns:
        member["dob"] = pd.to_datetime(member["dateofbirth"])
    elif "dob" in member.columns:
        member["dob"] = pd.to_datetime(member["dob"])
    elif "patient_dob" in member.columns:
        member["dob"] = pd.to_datetime(member["patient_dob"])

    member_id_col = (
        "uniquepersonkey"
        if "uniquepersonkey" in member.columns
        else ("patient_id" if "patient_id" in member.columns else "member_id")
    )

    if not claims.empty:
        claims["procedure_code"] = claims["procedure_code"].astype(str).str.strip()
        claims["line_service_date"] = pd.to_datetime(claims["line_service_date"])

        # Match procedure_code from CSV with code from lookup
        matched_claims = claims.merge(
            lookup,
            left_on="procedure_code",
            right_on="code",
            how="inner",
        )

        valid_claims = matched_claims[
            (matched_claims["line_service_date"] >= matched_claims["servicestartdate"])
            & (matched_claims["line_service_date"] <= matched_claims["serviceenddate"])
        ]
    else:
        valid_claims = pd.DataFrame()

    # -------------------------------------------------------------
    # 4. Evaluate Care Gaps
    # -------------------------------------------------------------
    unique_measures = (
        lookup[["measurename", "measuresource"]]
        .drop_duplicates()
        .to_dict(orient="records")
    )

    all_results = []
    year_end = pd.to_datetime(f"{measurement_year}-12-31")

    for target in unique_measures:
        m_name = target["measurename"]
        m_source = target["measuresource"]

        rules = lookup[
            (lookup["measurename"] == m_name)
            & (lookup["measuresource"] == m_source)
        ]

        if rules.empty:
            continue

        rule_gender = rules["gender"].iloc[0]
        min_age = rules["minage"].iloc[0]
        max_age = rules["maxage"].iloc[0]

        temp_member = member.copy()
        temp_member["calculated_age"] = (
            year_end - temp_member["dob"]
        ).dt.days // 365.25

        # Denominator Filter
        gender_mask = (
            (temp_member["gender"] == rule_gender)
            if rule_gender in ["M", "F"]
            else True
        )
        denom_df = temp_member[
            gender_mask
            & (temp_member["calculated_age"] >= min_age)
            & (temp_member["calculated_age"] <= max_age)
        ].copy()

        if denom_df.empty:
            continue

        # Numerator Check
        if not valid_claims.empty:
            num_members = valid_claims[
                (valid_claims["measurename"] == m_name)
                & (valid_claims["measuresource"] == m_source)
            ]["patient_id"].unique()
        else:
            num_members = []

        denom_df["measureName"] = m_name
        denom_df["measureSource"] = m_source
        denom_df["care_gap_status"] = denom_df[member_id_col].apply(
            lambda m_id: "CLOSED" if m_id in num_members else "OPEN"
        )
        denom_df["evaluation_date"] = datetime.today().strftime("%Y-%m-%d")

        all_results.append(denom_df)

    return (
        pd.concat(all_results, ignore_index=True)
        if all_results
        else pd.DataFrame()
    )

In [0]:

# -------------------------------------------------------------
# Execution Block
# -------------------------------------------------------------
if __name__ == "__main__":
    print("=== Interactive Notebook Test ===")

    print("Loading Gold Member and Bronze Measure Library tables...")
    member_df = spark.table("claimsprocessing.gold.gold_dimmember").toPandas()
    lookup_df = spark.table("claimsprocessing.bronze.measure_library").toPandas()

    claims_csv_path = "../../temp/837/superman.csv"

    if os.path.exists(claims_csv_path):
        print(f"Loading claims file: {claims_csv_path}")
        claims_df = pd.read_csv(claims_csv_path)
    else:
        print(f"Warning: {claims_csv_path} not found. Running with empty claims DataFrame.")
        claims_df = pd.DataFrame()

    print("Calculating care gaps...")
    final_report_df = calculate_care_gaps(
        member_df=member_df,
        claims_df=claims_df,
        lookup_df=lookup_df,
        measurement_year=2026,
    )

    print(f"\nExecution Complete! Evaluated {len(final_report_df)} record(s).")

    if not final_report_df.empty:
        display(final_report_df)
        output_file = "care_gap_report.csv"
        final_report_df.to_csv(output_file, index=False)
        print(f"Saved care gap report to: {output_file}")
    else:
        print("No care gaps evaluated. Check demographic filters or procedure code matches.")

In [0]:
def calculate_hedis_care_gaps(
    enrollment_df: pd.DataFrame,
    claims_df: pd.DataFrame,
    lookup_df: pd.DataFrame,
    measurement_year: int = 2026) -> pd.DataFrame:
    """
    Calculates Denominator, Numerator, and Open CareGaps for HEDIS/Core Sets.
    """
    if enrollment_df.empty or lookup_df.empty:
        return pd.DataFrame()

    # Normalize lookup_df columns to lower_snake_case internally
    lookup = lookup_df.copy()
    lookup.columns = lookup.columns.str.strip().str.lower()

    # 1. Convert Dates & Calculate Age as of Dec 31 of Measurement Year
    enrollment_df['dob'] = pd.to_datetime(enrollment_df['dob'])
    as_of_date = pd.to_datetime(f"{measurement_year}-12-31")
    enrollment_df['calculated_age'] = (as_of_date - enrollment_df['dob']).dt.days // 365

    if not claims_df.empty and 'service_date' in claims_df.columns:
        claims_df['service_date'] = pd.to_datetime(claims_df['service_date'])

    # Ensure continuous enrollment flag exists
    if 'continuous_enrollment_flag' not in enrollment_df.columns:
        enrollment_df['continuous_enrollment_flag'] = True

    # Deduplicate rules by measurecode
    distinct_lookup_df = lookup.drop_duplicates(subset=['measurecode'])

    denominator_matches = []

    # 2. Identify Denominator Matches
    for idx, rule in distinct_lookup_df.iterrows():
        min_age = rule.get('minage', 0)
        max_age = rule.get('maxage', 150)
        
        matching_patients = enrollment_df[
            (enrollment_df['calculated_age'] >= min_age) &
            (enrollment_df['calculated_age'] <= max_age) &
            (enrollment_df['continuous_enrollment_flag'] == True)
        ].copy()

        # Handle optional Gender rule safely
        gender_rule = rule.get('gender', 'ALL')
        if gender_rule != 'ALL' and 'gender' in matching_patients.columns:
            matching_patients = matching_patients[matching_patients['gender'].str.upper() == str(gender_rule).upper()]

        matching_patients['measureCode'] = rule['measurecode']
        matching_patients['measureName'] = rule['measurename']
        denominator_matches.append(matching_patients)

    if not denominator_matches:
        return pd.DataFrame()
    
    denominator_df = pd.concat(denominator_matches, ignore_index=True)

    # 3. Dynamic Numerator Matching (Qualifying Claims)
    complaint_pairs = set()
    if not claims_df.empty and 'claim_code' in claims_df.columns:
        claims_matched = claims_df.merge(
            lookup[['code', 'measurecode', 'eventname']],
            left_on='claim_code',
            right_on='code',
            how='inner'
        )
        
        id_col_claims = 'member_id' if 'member_id' in claims_matched.columns else 'patient_id'
        if not claims_matched.empty and id_col_claims in claims_matched.columns:
            complaint_pairs = set(zip(claims_matched[id_col_claims], claims_matched['measurecode']))

    # 4. Care Gap Evaluation
    id_col = 'member_id' if 'member_id' in denominator_df.columns else 'patient_id'

    denominator_df['care_gap_status'] = denominator_df.apply(
        lambda row: 'CLOSED' if (row[id_col], row['measureCode']) in complaint_pairs else 'OPEN',
        axis=1
    )

    output_cols = [id_col, 'gender', 'calculated_age', 'measureCode', 'measureName', 'care_gap_status']
    available_cols = [c for c in output_cols if c in denominator_df.columns]

    return denominator_df[available_cols]